# Homework 10: Fairness EDA of the Communities and Crime Dataset
### DSA4970/5970
### \<your name here\>

In this assignment, you will perform a fairness exploratory data analysis on the Communities and Crime dataset, located in the UCI Machine Learning Repository.  You can find the location and documentation of the dataset here:  https://archive.ics.uci.edu/dataset/183/communities+and+crime.  

Using the notebook from class as a guide, conduct a full EDA on the dataset.  Then answer the questions in the markdown sections at the bottom of this notebook skeleton.  Run your entire notebook and save it.  Then upload it to the assignment page for lesson 10.

__This assignment is worth 200 points in total.__

In [ ]:
# Check for the UCI Machine Learning Repository package and install if not present
import importlib
import subprocess
import sys
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

def ensure_ucimlrepo():
    try:
        importlib.import_module("ucimlrepo")
        print("ucimlrepo is already installed.")
    except ImportError:
        print("ucimlrepo not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ucimlrepo"])
        print("ucimlrepo installed successfully.")

ensure_ucimlrepo()


ucimlrepo is already installed.


In [ ]:
# ---------------------------------------------------------
# Load Communities & Crime dataset from UCI ML Repository
# ---------------------------------------------------------

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# Load dataset
communities = fetch_ucirepo(id=183)

# Extract the main dataframe
df = communities.data.original

# ---------------------------------------------------------
# Select subset of variables
# ---------------------------------------------------------

fairness_vars = [
    # Location
    "state",
    "county",

    # Demographics / protected-attribute-adjacent
    "racepctblack",
    "racePctWhite",
    "racePctAsian",
    "racePctHisp",

    # Socioeconomic structure
    "medIncome",
    "MedRent",
    "PctUnemployed",
    "PctNotHSGrad",

    # Housing / neighborhood structure
    "PctForeignBorn",

    # Policing intensity
    "PolicPerPop",

    # Crime outcomes
    "ViolentCrimesPerPop",

]

# Keep only the fairness variables
df_fairness = df[fairness_vars].copy()

# Set up lists of dimensions
geoglist = ['state', 'county']
senslist = ['racepctblack', 'racePctWhite', 'racePctAsian', 'racePctHisp']

# Set up a list of measurements
dfcols = df_fairness.columns.tolist()
measures = [col for col in dfcols if (col not in senslist and col not in geoglist)]

# Replace "?" with actual NaN values
df_fairness['county'] = df_fairness['county'].apply(pd.to_numeric, errors='coerce').astype('Int64')
df_fairness[measures] = df_fairness[measures].apply(pd.to_numeric, errors='coerce')

# Preview
print(df_fairness.head())
print(df_fairness.shape)


   state county  racepctblack  racePctWhite  racePctAsian  racePctHisp  \
0      8    NaN          0.02          0.90          0.12         0.17   
1     53    NaN          0.12          0.74          0.45         0.07   
2     24    NaN          0.49          0.56          0.17         0.04   
3     34      5          1.00          0.08          0.12         0.10   
4     42     95          0.02          0.95          0.09         0.05   

   medIncome  MedRent  PctUnemployed  PctNotHSGrad  PctForeignBorn  \
0       0.37     0.34           0.27          0.18            0.12   
1       0.31     0.37           0.27          0.24            0.21   
2       0.30     0.31           0.36          0.43            0.14   
3       0.58     0.89           0.33          0.25            0.19   
4       0.50     0.38           0.12          0.30            0.11   

  PolicPerPop  ViolentCrimesPerPop  
0        0.13                 0.20  
1         NaN                 0.67  
2         NaN          

#### Code (60 points)
Build your fairness EDA by adding code blocks to the notebook below this markdown section. Run all the relevant analyses.  Use the instructional_eda.ipynb notebook as a guide. Run your entire script to produce the outputs.

In [ ]:
"""
Determine descriptive statistics for the sensitive attributes

Start by checking the descriptive statistics for the sensitive attributes.  You will see the variation
in the racial composition of the communities in the dataset.  Next, create a new column called 'majority_group'
that identifies the majority racial group in each community.  Use this categorical variable instead of the racial
percentages in your further analyses unless otherwise stated.

Some hints:
+ The measurement variables are all continuous.  So what test should you use when comparing distributions across groups?
+ You need to conduct significance tests for each pair of groups values.
+ You will need to use the Bonferroni correction to adjust for multiple distribution comparisons.  How many comparisons are you making?

""" 

# Begin your code here


#### Q1 (20 points): What is the sensitive attribute in this dataset?
__Type your answer below:__

#### Q2 (20 points): From your stratified distribution analysis, which variables have a statistically significant difference by sensitive attribute?
__Type your answer below:__

#### Q3 (20 points): Which measurement variable has the highest level of missingness?
__Type your answer below:__

#### Q4 (20 points): Is this variable MCAR, MAR, or MNAR?  Explain your answer.
__Type your answer below:__

#### Q5 (20 points): To look for correlations or associations within one heatmap, all variables must be numerical (correlation) or all must be categorical (association).  Because we have numerical values for the percentages of racial groups in each neighborhood (e.g. racepctblack), we can use those instead of majority_group with our measures to create a set of only numerical variables.  What statistic do we use in this case to generate our heatmap?
__Type your answer below:__

#### Q6 (20 points): Produce a heatmap with all numerical variables.  Drop the highest missingness measurement variable so you can keep most of your rows.  What measures are related to your sensitive attribute(s)?  Which one has the greatest degree of correlation or association to a sensitive attribute?  What is the value of its correlation or association and to what sensitive attribute is it related?
__Type your answer below:__

#### Q7 (20 points): Consider violent crimes per population as the target variable. Is there a conditional outcome difference based upon the sensitive attribute(s)? Explain how you know if there is or is not.  To answer this question, perform a regression using only numeric variables.  Don't use majority_group -- you can simplify the regression by avoiding one-hot encoding by sticking with numeric variables.  Also, exclude the measure with the highest missingness -- OLS regression cannot handle missing values.
__Type your answer below:__